# 🕵️‍♂️ Día 25/04: Revisión de Datos y Preprocesado Inicial
**Objetivo de este notebook:**
1. Cargar los datos brutos de la competición.
2. Hacer un "Sanity Check" para detectar problemas (nulos, desbalance, duplicados).
3. Definir y aplicar la función de preprocesado unificada que usaremos todo el equipo.

In [1]:
import pandas as pd
import re

# Configuración visual para ver bien los textos largos
pd.set_option('display.max_colwidth', None)

## 1. Carga de Datos
Asegúrate de tener los archivos `train.csv` y `test_nolabel.csv` en la misma carpeta que este notebook.

In [2]:
print("Cargando datos...")
train = pd.read_csv("../../data/train.csv")
test = pd.read_csv("../../data/test_nolabel.csv")

print("¡Datos cargados!")

Cargando datos...
¡Datos cargados!


## 2. Radiografía de los Datos (Sanity Check)
Vamos a revisar el estado real del dataset para anotar los problemas que debemos tener en cuenta durante el modelado.

In [3]:
print("="*40)
print(" ANÁLISIS DEL DATASET DE ENTRENAMIENTO")
print("="*40)

print(f"-> Dimensiones: {train.shape[0]} filas y {train.shape[1]} columnas")

# 1. Desbalance de la variable objetivo
print("\n-> Desbalance de la variable objetivo ('label'):")
print(train['label'].value_counts(normalize=True).round(4) * 100) 

# 2. Valores Nulos
print("\n-> Valores Nulos por columna:")
nulos = train.isnull().sum()
print(nulos[nulos > 0])

# 3. Duplicados exactos
duplicados = train.duplicated().sum()
print(f"\n-> Filas exactamente duplicadas: {duplicados}")

 ANÁLISIS DEL DATASET DE ENTRENAMIENTO
-> Dimensiones: 8950 filas y 8 columnas

-> Desbalance de la variable objetivo ('label'):
label
1    64.75
0    35.25
Name: proportion, dtype: float64

-> Valores Nulos por columna:
speaker_job    2482
state_info     1930
dtype: int64

-> Filas exactamente duplicadas: 0


### ⚠️ Problemas detectados (Para el acta de la reunión)
* **Desbalance:** Hay una proporción de ~64% (clase 1) vs ~35% (clase 0). Necesitaremos usar `class_weight='balanced'` en los modelos o métricas como el F1-Score.
* **Nulos:** Faltan datos en `speaker_job` y `state_info`. Los imputaremos como "unknown".
* **Ruido en texto:** El texto libre tiene mayúsculas, signos de puntuación y la columna de temas (`subject`) está separada por comas.

## 3. Función Maestra de Preprocesado
Esta es la función que **todo el equipo debe importar y usar** antes de entrenar sus respectivos modelos para unificar criterios.

In [4]:
def preprocesar_datos(df):
    """
    Limpia el DataFrame estandarizando texto, rellenando nulos y 
    preparando las categorías para los modelos.
    """
    df_limpio = df.copy()
    
    # A. Tratamiento de nulos
    columnas_con_nulos = ['speaker_job', 'state_info', 'speaker', 'party_affiliation', 'subject']
    for col in columnas_con_nulos:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna('unknown')
            
    # B. Limpieza de categorías (Minúsculas y sin espacios extra)
    if 'party_affiliation' in df_limpio.columns:
        df_limpio['party_affiliation'] = df_limpio['party_affiliation'].str.lower().str.strip()
    if 'speaker' in df_limpio.columns:
        df_limpio['speaker'] = df_limpio['speaker'].str.lower().str.strip()
        
    # C. Arreglar la columna 'subject' (Comas a espacios)
    if 'subject' in df_limpio.columns:
        df_limpio['subject'] = df_limpio['subject'].astype(str).str.replace(',', ' ')
        
    # D. Limpieza profunda del texto principal ('statement')
    def clean_text(texto):
        if not isinstance(texto, str): return ""
        texto = texto.lower() 
        texto = re.sub(r'\[.*?\]', '', texto) 
        texto = re.sub(r'[^a-z0-9\s-]', '', texto) 
        texto = re.sub(r'\s+', ' ', texto).strip() 
        return texto

    if 'statement' in df_limpio.columns:
        df_limpio['statement_clean'] = df_limpio['statement'].apply(clean_text)
        
    return df_limpio

## 4. Aplicación de la limpieza
Eliminamos los duplicados del conjunto de entrenamiento (nunca del de test) y aplicamos nuestra función estandarizada.

In [5]:
# Eliminar duplicados solo en train
if duplicados > 0:
    train = train.drop_duplicates()
    print(f"Borradas {duplicados} filas duplicadas de Train.")

# Limpiar ambos datasets
train_clean = preprocesar_datos(train)
test_clean = preprocesar_datos(test)

print("¡Datasets listos para el Sprint de Modelado!")

# Veamos el resultado en la primera fila
print("\n--- ANTES ---")
print(train['statement'].iloc[0])
print("\n--- DESPUÉS ---")
print(train_clean['statement_clean'].iloc[0])

¡Datasets listos para el Sprint de Modelado!

--- ANTES ---
China is in the South China Sea and (building)a military fortress the likes of which perhaps the world has not seen.

--- DESPUÉS ---
china is in the south china sea and buildinga military fortress the likes of which perhaps the world has not seen
